In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
# doc du lieu 
sales = pd.read_csv("sales.csv")
sample_submission = pd.read_csv("sample_submission.csv")
sales["Date"] = pd.to_datetime(sales["Date"])
sample_submission["Date"] = pd.to_datetime(sample_submission["Date"])
#sort du lieu
sales = sales.sort_values("Date").reset_index(drop=True)
sample_submission = sample_submission.sort_values("Date").reset_index(drop=True)
# kiem tra du lieu
print(sales.head())
print(sales.tail())
print(sales.shape)
print(sample_submission.head())
print(sample_submission.shape)
#tao ban sao
df = sales.copy()
# tao calendar
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["day"] = df["Date"].dt.day
df["dayofweek"] = df["Date"].dt.dayofweek
df["quarter"] = df["Date"].dt.quarter
df["weekofyear"] = df["Date"].dt.isocalendar().week.astype(int)
df["is_month_start"] = df["Date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["Date"].dt.is_month_end.astype(int)
# tao lag features
for lag in [1, 7, 14, 28, 30]:
    df[f"Revenue_lag_{lag}"] = df["Revenue"].shift(lag)
# tao rolling features
for window in [7, 14, 28]:
		df[f"Revenue_roll_mean_{window}"] = df["Revenue"].shift(1).rolling(window).mean()
		df[f"Revenue_roll_std_{window}"] = df["Revenue"].shift(1).rolling(window).std()
# chon feature and target
feature_cols = [
    "year", "month", "day", "dayofweek", "quarter", "weekofyear",
    "is_month_start", "is_month_end",
    "Revenue_lag_1", "Revenue_lag_7", "Revenue_lag_14", "Revenue_lag_28", "Revenue_lag_30",
    "Revenue_roll_mean_7", "Revenue_roll_mean_14", "Revenue_roll_mean_28",
    "Revenue_roll_std_7", "Revenue_roll_std_14", "Revenue_roll_std_28"
]
# tao rolling features
for window in [7, 14, 28, 30]:
		df[f"Revenue_roll_mean_{window}"] = df["Revenue"].shift(1).rolling(window).mean()
		df[f"Revenue_roll_std_{window}"] = df["Revenue"].shift(1).rolling(window).std()

# Tạo df_model sau khi có tất cả các features
df_model = df.dropna().copy()

# Định nghĩa hàm để chọn features
def new_func(feature_cols, data):
    return data[feature_cols]

x = df[feature_cols]
y = df["Revenue"]
# chia train 
train_data = df_model[df_model["Date"] < "2022-01-01"].copy()
valid_data = df_model[df_model["Date"] >= "2022-01-01"].copy()

x_train = new_func(feature_cols, train_data)
y_train = train_data["Revenue"]

x_valid = new_func(feature_cols, valid_data)
y_valid = valid_data["Revenue"]
# train baseline model 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(x_train, y_train)
valid_pred = model.predict(x_valid)
mae = mean_absolute_error(y_valid, valid_pred)
rmse = mean_squared_error(y_valid, valid_pred) ** 0.5
r2 = r2_score(y_valid, valid_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
plt.figure(figsize=(14, 5))
plt.plot(valid_data["Date"], y_valid, label="Actual Revenue")
plt.plot(valid_data["Date"], valid_pred, label="Predicted Revenue")
plt.title("Validation: Actual vs Predicted Revenue")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.show()

        Date     Revenue        COGS
0 2012-07-04  5123547.94  3982991.19
1 2012-07-05  2751773.45  2150580.23
2 2012-07-06  3054029.42  2517632.84
3 2012-07-07  2667930.94  2108246.62
4 2012-07-08  2360851.90  1808622.79
           Date     Revenue        COGS
3828 2022-12-27  2100553.66  2184872.24
3829 2022-12-28  3448729.20  3513621.00
3830 2022-12-29  3083944.33  3170787.10
3831 2022-12-30  2884668.76  3022292.15
3832 2022-12-31  2383037.48  2279288.13
(3833, 3)
        Date     Revenue        COGS
0 2023-01-01  2665507.20  2518885.15
1 2023-01-02  1280007.89  1136463.00
2 2023-01-03  1015899.51   822721.12
3 2023-01-04  1142997.27   914554.18
4 2023-01-05  1236312.34   984390.24
(548, 3)
MAE: 575114.030318767
RMSE: 823727.2774105351
R2: 0.7578108820628815
